In [1]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# ==========================================
# 1. SETUP AMBIENTE E CARICAMENTO DATI
# ==========================================
# Sostituisci "input_dataset.csv" con il percorso reale del tuo file
INPUT_CSV = "df_duplicati.csv"
OUTPUT_CSV_ANONIMO = "dataset_report_anonimizzati_Llama-3.2-3B.csv"

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Il file '{INPUT_CSV}' non esiste.")

df_input = pd.read_csv(INPUT_CSV)

if "testo_originale" not in df_input.columns or "nome_file" not in df_input.columns:
    raise ValueError("Il DataFrame sorgente deve contenere le colonne 'testo_originale' e 'nome_file'.")

# Checkpoint ottimizzato: lettura singola pre-ciclo $O(1)$ lookup
file_processati = set()
if os.path.exists(OUTPUT_CSV_ANONIMO):
    df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
    if "nome_file" in df_check.columns:
        file_processati = set(df_check['nome_file'].values)

# ==========================================
# 2. ESTRAZIONE METADATI E PULIZIA TESTO
# ==========================================
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')
        fine_periodo = periodo_match.group(2).replace('_', ' ')
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base

    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,
        "fine_periodo": fine_periodo,
        "nome_file": nome_file
    }
df_input

/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Unnamed: 0,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale,gruppo,Llama_3_1_8B_anonimo,Llama_3_1_8B_anonimo_promtp_in_context,Llama_3_2_3B_anonimo
0,7,El_Salvador_Mar_2025_-_Feb_2026_KeyResults.txt,El Salvador,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
1,9,Honduras_Mar_2025_-_Feb_2026_KeyResults.txt,Honduras,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
2,23,Guatemala_Mar_2025_-_Feb_2026_KeyResults.txt,Guatemala,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
3,31,El_Salvador_Jun_2020_-_Aug_2020_KeyResults.txt,El Salvador,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
4,35,Honduras_Jun_2020_-_Aug_2020_KeyResults.txt,Honduras,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
5,59,Guatemala_Jun_2020_-_Aug_2020_KeyResults.txt,Guatemala,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
6,68,El_Salvador_Jun_2022_-_Aug_2022_KeyResults.txt,El Salvador,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
7,70,Honduras_Jun_2022_-_Aug_2022_KeyResults.txt,Honduras,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
8,74,Guatemala_Jun_2022_-_Aug_2022_KeyResults.txt,Guatemala,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
9,162,El_Salvador_Nov_2018_-_Apr_2019_KeyResults.txt,El Salvador,Nov 2018 / Apr 2019,Nov 2018,Apr 2019,The Tri-national Border Federation of Río Lemp...,3.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...


In [2]:


# ==========================================
# 3. DOWNLOAD GGUF E INIZIALIZZAZIONE LLAMA.CPP
# ==========================================
print("Recupero dei pesi quantizzati GGUF (Llama-3.1-8B-Instruct, 8-bit)...")
modello_gguf_path = hf_hub_download(
    repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
)

print("Inizializzazione del motore di inferenza MPS (Metal)...")
llm = Llama(
    model_path=modello_gguf_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False
)

# ==========================================
# 4. GESTIONE DEI PROMPT SEMANTICI
# ==========================================
# Regola 5 eliminata come richiesto
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

# ==========================================
# (Inserisci questo blocco prima del ciclo FOR)
# ==========================================
nome_modello = "Llama_3_1_8B_8bit"
colonna_output = f"{nome_modello}_anonimo"

# Inizializza la colonna nel DataFrame se non esiste per evitare KeyError
if colonna_output not in df_input.columns:
    df_input[colonna_output] = pd.NA

print(f"\nTrovati {len(df_input)} report nel DataFrame da elaborare...\n")

# ==========================================
# 5. ELABORAZIONE DEL BATCH DAL DATAFRAME
# ==========================================
for index, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Elaborazione report"):

    # Lookup vettoriale per evitare inferenze ridondanti:
    # Se la colonna contiene già un valore per questa riga, salta.
    if pd.notna(row.get(colonna_output)):
        continue

    nome_file = row['nome_file']
    testo_originale = str(row['testo_originale'])
    testo_pulito = re.sub(r'^.*?={20,}\n*', '', testo_originale, flags=re.DOTALL).strip()

    if not testo_pulito or testo_pulito.lower() == 'nan':
        print(f" -> Avviso: il testo per {nome_file} risulta vuoto dopo la pulizia.")
        continue

    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
    ]

    try:
        outputs = llm.create_chat_completion(
            messages=messages,
            max_tokens=700,
            temperature=0.1
        )

        scheda_anonima = outputs["choices"][0]["message"]["content"].strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Inserimento atomico del risultato nel DataFrame in RAM
        df_input.loc[index, colonna_output] = scheda_anonima

    except Exception as e:
        print(f"\nErrore di esecuzione sul file {nome_file}: {e}")
        time.sleep(2)



Recupero dei pesi quantizzati GGUF (Llama-3.1-8B-Instruct, 8-bit)...


Inizializzazione del motore di inferenza MPS (Metal)...

Trovati 35 report nel DataFrame da elaborare...



Elaborazione report: 100%|██████████| 35/35 [11:14<00:00, 19.29s/it]


In [3]:
# ==========================================
# 6. ESPORTAZIONE FINALE (OBBLIGATORIA)
# ==========================================
# Sovrascrive il file originale (o ne crea uno nuovo) con la colonna aggiunta
df_input.to_csv(INPUT_CSV, index=False, encoding='utf-8')
print(f"\nPipeline completata. DataFrame salvato in: '{INPUT_CSV}' con la nuova colonna '{colonna_output}'.")


Pipeline completata. DataFrame salvato in: 'df_duplicati.csv' con la nuova colonna 'Llama_3_1_8B_8bit_anonimo'.
